In [1]:
import pickle

import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [8]:
from sklearn.pipeline import make_pipeline

In [3]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("green-taxi-duration")

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1784746229985, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1784746229985, lifecycle_stage='active', name='green-taxi-duration', tags={}, trace_location=None, workspace='default'>

In [4]:
def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df


def prepare_dictionaries(df: pd.DataFrame):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    return dicts

In [5]:
df_train = read_dataframe('../../data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('../../data/green_tripdata_2021-02.parquet')

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

dict_train = prepare_dictionaries(df_train)
dict_val = prepare_dictionaries(df_val)

In [9]:
with mlflow.start_run() as mlflow_run:
    params = dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=0)
    mlflow.log_params(params)

    dv = DictVectorizer()
    model = RandomForestRegressor(**params, n_jobs=-1)
    pipeline = make_pipeline(dv, model)

    pipeline.fit(dict_train, y_train)
    y_pred = pipeline.predict(dict_val)
    
    rmse = root_mean_squared_error(y_pred, y_val)
    print(params, rmse)
    mlflow.log_metric('rmse', rmse)
    
    # bundle it into the model's own artifact dir at log time
    mlflow.sklearn.log_model(
        pipeline,
        name="model"
    )

2026/07/22 17:17:53 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/irb/Documents/mlops-zoomcamp/04-deployment/web-service-mlflow


{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 0} 6.7558229919200725


2026/07/22 17:17:53 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/irb/Documents/mlops-zoomcamp/04-deployment/web-service-mlflow
2026/07/22 17:17:53 INFO mlflow.utils.environment: Detected uv project at /home/irb/Documents/mlops-zoomcamp/04-deployment/web-service-mlflow. Attempting to export requirements via 'uv export'.
2026/07/22 17:17:53 INFO mlflow.utils.uv_utils: Exported 89 dependencies via uv
2026/07/22 17:17:53 INFO mlflow.utils.environment: Successfully exported 89 requirements from uv project. Skipping package capture based inference.
2026/07/22 17:17:53 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - aiohttp (current: 3.14.1, required: aiohttp==3.14.2; sys_platform == "linux")
 - alembic (current: 1.18.4, required: alembic==1.18.5; sys_platform == "linux")
 - anyio (current: 4.13.0, required: anyio==4.14.2; sys_platform == "linux")
 - c

🏃 View run delightful-calf-251 at: http://localhost:5000/#/experiments/2/runs/38ee7005f2d64eff8f1e2edad80cd244
🧪 View experiment at: http://localhost:5000/#/experiments/2
